# Pose Auto-Label Harvest — Preds as Editable GT

**Purpose:** generate YOLO-pose labels from `yolo11x-pose.pt` preds so you only correct misses.

1. Edit `AUTO_LABEL_SOURCES` in the next cell — add `(video, ranges)` entries.
2. Run the Runner cell → it writes `datasets/auto_label_v1` (`images/train/*.jpg` + `labels/train/*.txt` + `data.yaml`).
3. `zip datasets/auto_label_v1` → upload into existing Platform dataset `vid26` → correct in editor (especially bottom-left girl).

Leave `AUTO_LABEL_SOURCES = []` to skip safely. `PRE_LABEL_CONF=0.08` surfaces weak persons for review.
`flip_idx` is pulled live from `ultralytics/cfg/datasets/coco8-pose.yaml` — never hand-typed.

In [13]:
### Auto-label harvest — CONFIG (generic, edit sources below)
from pathlib import Path
import cv2
import yaml as _yaml
import shutil
from ultralytics import YOLO as _YOLO2
import ultralytics as _ul

AUTO_LABEL_SOURCES = [
    ("video/input/vid33.mp4", None),                    ### None = whole video sampled
    ("video/input/vid26.mp4", None),                    ### None = whole video sampled

    # ("video/input/vid12.mp4", [(30, 120)]),             ### or specific (start_s, end_s) ranges
]
FRAME_EVERY_S  = 2        ### standard density: ~1 frame / 5s  (~60-100 frames)
PRE_LABEL_CONF = 0.08     ### LOW: weak persons surface for your review
AUTO_OUT       = Path("datasets") / "auto_label_v1"
MODEL_BASE     = "yolo11x-pose.pt"
IMGSZ          = 640


In [ ]:
# shutil.rmtree(AUTO_OUT) # Uncomment to delete dataset

_ref = _yaml.safe_load(open(_ul.__file__.replace('__init__.py','') + "/cfg/datasets/coco8-pose.yaml", encoding="utf-8"))
FLIP_IDX = _ref.get("flip_idx", [0,2,1,3,4,6,5,8,7,10,9,12,11,14,13,16,15])
_m = _YOLO2(MODEL_BASE)
_img_dir = AUTO_OUT / "images" / "train"
_lbl_dir = AUTO_OUT / "labels" / "train"
_img_dir.mkdir(parents=True, exist_ok=True); _lbl_dir.mkdir(parents=True, exist_ok=True)
_written=0

for _vid,_rngs in AUTO_LABEL_SOURCES:
    _cap=cv2.VideoCapture(_vid)
    assert _cap.isOpened(), _vid
    _fps=_cap.get(cv2.CAP_PROP_FPS) or 25.0
    _total=int(_cap.get(cv2.CAP_PROP_FRAME_COUNT))
    _spans=_rngs or [(0, _total/_fps)]
    _stem=Path(_vid).stem
    for _s0,_s1 in _spans:
        _t=_s0
        while _t <= min(_s1, _total/_fps):
            _cap.set(cv2.CAP_PROP_POS_MSEC, _t*1000)
            _ok,_fr=_cap.read()
            if not _ok: break
            _r=_m.predict(_fr, conf=PRE_LABEL_CONF, classes=[0], imgsz=IMGSZ, verbose=False)[0]
            _name=f"{_stem}_{int(_t*1000):07d}"
            cv2.imwrite(str(_img_dir/f"{_name}.jpg"), _fr)
            _lines=[]
            for _bi in range(len(_r.boxes)):
                _x1,_y1,_x2,_y2=_r.boxes.xyxy[_bi].tolist()
                _W,_H=_fr.shape[1],_fr.shape[0]
                _cx,_cy,_bw,_bh=( _x1+_x2)/2/_W,( _y1+_y2)/2/_H,(_x2-_x1)/_W,(_y2-_y1)/_H
                _kc=_r.keypoints.conf[_bi] if _r.keypoints is not None else None
                _parts=[f"0 {_cx:.6f} {_cy:.6f} {_bw:.6f} {_bh:.6f}"]
                for _ki,( _kx,_ky) in enumerate(_r.keypoints.xyn[_bi].tolist()):
                    # Fixed: v=2 for visible (conf>0.5), v=1 for occluded (conf>0.1), v=0 for not detected
                    if _kc is not None and _kc[_ki] > 0.5:
                        _v = 2  # Visible
                    elif _kc is not None and _kc[_ki] > 0.1:
                        _v = 1  # Occluded/low confidence
                    else:
                        _v = 0  # Not detected
                    _parts.append(f"{_kx:.6f} {_ky:.6f} {_v}")
                _lines.append(" ".join(_parts))
            (_lbl_dir/f"{_name}.txt").write_text("\n".join(_lines))
            _written+=1; _t+=FRAME_EVERY_S
    _cap.release()
(_lbl_dir.parent.parent/"data.yaml").write_text(f"task: pose\nnames:\n  0: Person\nkpt_shape: [17, 3]\nflip_idx: {FLIP_IDX}\ntrain: images/train\nval: images/train\n")

zip_path = shutil.make_archive(str(AUTO_OUT), 'zip', str(AUTO_OUT))